# Titanic 
## Score: .79904

In [8]:
import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    XGBClassifier = None
    HAS_XGB = False

SEED = 42
np.random.seed(SEED)
ROOT = Path.cwd()
DATA = ROOT / "titanic"
RARE = {"Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", "Jonkheer", "Dona"}


In [9]:
def build_xy(train_df: pd.DataFrame, test_df: pd.DataFrame):
    y = train_df["Survived"].to_numpy()
    tr = train_df.drop(columns=["Survived"]).copy()
    te = test_df.copy()

    ticket_vc = pd.concat([tr["Ticket"], te["Ticket"]], ignore_index=True).value_counts()
    med_fare = tr["Fare"].median()
    mode_emb = tr["Embarked"].mode().iloc[0]

    for pcl in (1, 2, 3):
        m = tr.loc[tr["Pclass"] == pcl, "Fare"].median()
        if pd.notna(m):
            tr.loc[tr["Pclass"] == pcl, "Fare"] = tr.loc[tr["Pclass"] == pcl, "Fare"].fillna(m)
            te.loc[te["Pclass"] == pcl, "Fare"] = te.loc[te["Pclass"] == pcl, "Fare"].fillna(m)
    tr["Fare"] = tr["Fare"].fillna(med_fare)
    te["Fare"] = te["Fare"].fillna(med_fare)
    tr["Embarked"] = tr["Embarked"].fillna(mode_emb)
    te["Embarked"] = te["Embarked"].fillna(mode_emb)

    def impute_age(ref: pd.DataFrame, df: pd.DataFrame) -> None:
        gmed = ref["Age"].median()
        for (sex, pcl), m in ref.groupby(["Sex", "Pclass"])["Age"].median().items():
            mm = m if pd.notna(m) else gmed
            msk = (df["Sex"] == sex) & (df["Pclass"] == pcl) & df["Age"].isna()
            df.loc[msk, "Age"] = mm
        df["Age"] = df["Age"].fillna(gmed)

    impute_age(tr, tr)
    impute_age(tr, te)

    def feats(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        pcl = pd.to_numeric(out["Pclass"], errors="coerce").fillna(3).astype(int)
        out["Title"] = out["Name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()
        out["Title"] = out["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
        out["Title"] = out["Title"].where(~out["Title"].isin(RARE), "Rare")
        out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
        out["IsAlone"] = (out["FamilySize"] == 1).astype(int)
        out["SexFemale"] = (out["Sex"] == "female").astype(int)
        out["IsChild"] = (out["Age"] < 16).astype(int)
        out["FemaleOrChild"] = ((out["Sex"] == "female") | (out["Age"] < 16)).astype(int)
        out["Deck"] = out["Cabin"].apply(lambda x: str(x)[0] if pd.notna(x) and str(x) and str(x)[0].isalpha() else "U")
        out["TicketGroupSize"] = out["Ticket"].map(ticket_vc).fillna(1).astype(int)
        out["FarePerPerson"] = out["Fare"] / out["FamilySize"].clip(lower=1)
        out["HasCabin"] = out["Cabin"].notna().astype(int)
        out["AgePclass"] = out["Age"] * pcl.astype(float)
        out["AgeBin"] = pd.cut(out["Age"], bins=[0, 12, 18, 35, 60, 200], labels=["c", "t", "y", "a", "s"]).astype(str)
        out["Surname"] = out["Name"].str.split(",").str[0].str.strip()
        out["TicketPrefix"] = out["Ticket"].astype(str).str.replace(r"[0-9./ ]", "", regex=True)
        out["TicketPrefix"] = out["TicketPrefix"].replace("", "NONE")
        out = out.drop(columns=["Name", "Ticket", "Cabin", "PassengerId"], errors="ignore")
        out["Pclass"] = out["Pclass"].astype(str)
        for c in ["Sex", "Embarked", "Title", "Deck", "AgeBin", "Surname", "TicketPrefix"]:
            out[c] = out[c].astype(str)
        return out

    X = feats(tr)
    X_test = feats(te)
    return X, y, X_test


def add_oof_target_encoding(X: pd.DataFrame, y: np.ndarray, X_test: pd.DataFrame, cols, n_splits=5, alpha=15.0):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    y_s = pd.Series(y)
    global_mean = y_s.mean()
    X_new = X.copy()
    X_test_new = X_test.copy()

    for col in cols:
        oof_col = np.zeros(len(X), dtype=float)
        for tr_i, va_i in cv.split(X, y):
            tr_c = X.iloc[tr_i][col]
            y_tr = y_s.iloc[tr_i]
            stat = y_tr.groupby(tr_c).agg(["mean", "count"])
            smooth = (stat["mean"] * stat["count"] + global_mean * alpha) / (stat["count"] + alpha)
            oof_col[va_i] = X.iloc[va_i][col].map(smooth).fillna(global_mean).to_numpy()
        full_stat = y_s.groupby(X[col]).agg(["mean", "count"])
        full_smooth = (full_stat["mean"] * full_stat["count"] + global_mean * alpha) / (full_stat["count"] + alpha)
        X_new[col + "_te"] = oof_col
        X_test_new[col + "_te"] = X_test[col].map(full_smooth).fillna(global_mean).to_numpy()
    return X_new, X_test_new


train_raw = pd.read_csv(DATA / "train.csv")
test_raw = pd.read_csv(DATA / "test.csv")
pid = test_raw["PassengerId"].values
X, y, X_test = build_xy(train_raw, test_raw)
cat_cols = ["Sex", "Embarked", "Title", "Deck", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
te_cols = ["Title", "Deck", "Embarked", "Pclass", "AgeBin", "Surname", "TicketPrefix"]
X_te, X_test_te = add_oof_target_encoding(X, y, X_test, te_cols, n_splits=5, alpha=15.0)

hgb_features = [
    "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone", "AgePclass",
    "TicketGroupSize", "FarePerPerson", "HasCabin",
    "SexFemale", "IsChild", "FemaleOrChild",
] + [c + "_te" for c in te_cols]

imp = SimpleImputer(strategy="median")
X_hgb = imp.fit_transform(X_te[hgb_features])
X_test_hgb = imp.transform(X_test_te[hgb_features])


In [10]:
def run_cv(seed_offset: int = 0):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + seed_offset)
    oof_cb1 = np.zeros(len(y), dtype=float)
    oof_cb2 = np.zeros(len(y), dtype=float)
    oof_hgb = np.zeros(len(y), dtype=float)
    oof_et = np.zeros(len(y), dtype=float)
    p_cb1_t = np.zeros(len(X_test), dtype=float)
    p_cb2_t = np.zeros(len(X_test), dtype=float)
    p_hgb_t = np.zeros(len(X_test), dtype=float)
    p_et_t = np.zeros(len(X_test), dtype=float)
    oof_xgb = np.zeros(len(y), dtype=float)
    p_xgb_t = np.zeros(len(X_test), dtype=float)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        Xc_tr, Xc_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        Xh_tr, Xh_va = X_hgb[tr_idx], X_hgb[va_idx]

        cb1 = CatBoostClassifier(
            iterations=20000,
            learning_rate=0.016,
            depth=7,
            l2_leaf_reg=3.0,
            random_seed=SEED + seed_offset + fold,
            thread_count=1,
            verbose=False,
        )
        cb1.fit(Xc_tr, y_tr, cat_features=cat_cols, eval_set=(Xc_va, y_va), early_stopping_rounds=320, verbose=False)

        cb2 = CatBoostClassifier(
            iterations=26000,
            learning_rate=0.012,
            depth=9,
            l2_leaf_reg=4.5,
            random_seed=SEED + 100 + seed_offset + fold,
            thread_count=1,
            verbose=False,
        )
        cb2.fit(Xc_tr, y_tr, cat_features=cat_cols, eval_set=(Xc_va, y_va), early_stopping_rounds=380, verbose=False)

        hgb = HistGradientBoostingClassifier(
            learning_rate=0.02,
            max_iter=1400,
            max_depth=8,
            min_samples_leaf=6,
            l2_regularization=0.35,
            random_state=SEED + seed_offset + fold,
        )
        hgb.fit(Xh_tr, y_tr)

        et = ExtraTreesClassifier(
            n_estimators=900,
            max_depth=20,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=SEED + 17 * seed_offset + fold,
            n_jobs=-1,
        )
        et.fit(Xh_tr, y_tr)

        p1 = cb1.predict_proba(Xc_va)[:, 1]
        p2 = cb2.predict_proba(Xc_va)[:, 1]
        p3 = hgb.predict_proba(Xh_va)[:, 1]
        p4 = et.predict_proba(Xh_va)[:, 1]
        oof_cb1[va_idx], oof_cb2[va_idx], oof_hgb[va_idx], oof_et[va_idx] = p1, p2, p3, p4

        p_cb1_t += cb1.predict_proba(X_test)[:, 1] / cv.n_splits
        p_cb2_t += cb2.predict_proba(X_test)[:, 1] / cv.n_splits
        p_hgb_t += hgb.predict_proba(X_test_hgb)[:, 1] / cv.n_splits
        p_et_t += et.predict_proba(X_test_hgb)[:, 1] / cv.n_splits

        if HAS_XGB:
            xgb = XGBClassifier(
                n_estimators=5000,
                learning_rate=0.018,
                max_depth=4,
                min_child_weight=2,
                subsample=0.86,
                colsample_bytree=0.78,
                reg_lambda=1.7,
                reg_alpha=0.12,
                random_state=SEED + 31 * seed_offset + fold,
                eval_metric="logloss",
                early_stopping_rounds=120,
                n_jobs=-1,
            )
            xgb.fit(Xh_tr, y_tr, eval_set=[(Xh_va, y_va)], verbose=False)
            oof_xgb[va_idx] = xgb.predict_proba(Xh_va)[:, 1]
            p_xgb_t += xgb.predict_proba(X_test_hgb)[:, 1] / cv.n_splits

    parts_oof = [oof_cb1, oof_cb2, oof_hgb]
    parts_te = [p_cb1_t, p_cb2_t, p_hgb_t]
    if HAS_XGB:
        parts_oof.append(oof_xgb)
        parts_te.append(p_xgb_t)
    parts_oof.append(oof_et)
    parts_te.append(p_et_t)
    stack_oof = np.column_stack(parts_oof)
    stack_te = np.column_stack(parts_te)
    n_meta = stack_oof.shape[1]

    best_acc, best_meta, best_t = -1.0, None, 0.5
    for C in (0.12, 0.2, 0.32, 0.5, 0.75, 1.1, 1.6, 2.4, 3.5):
        meta = LogisticRegression(max_iter=10000, C=C, random_state=SEED)
        meta.fit(stack_oof, y)
        m_oof = meta.predict_proba(stack_oof)[:, 1]
        for t in np.linspace(0.36, 0.64, 141):
            a = accuracy_score(y, (m_oof >= t).astype(int))
            if a > best_acc:
                best_acc, best_meta, best_t = a, meta, float(t)
    meta = best_meta
    m_oof = meta.predict_proba(stack_oof)[:, 1]
    test_prob = meta.predict_proba(stack_te)[:, 1]
    return {
        "mode": f"meta{n_meta}",
        "acc": best_acc,
        "t": best_t,
        "meta": meta,
        "oof_prob": m_oof,
        "test_prob": test_prob,
    }


safe = run_cv(seed_offset=0)
print("cv", safe["mode"], "acc", round(safe["acc"], 5), "t", round(safe["t"], 3))
print("meta_coef", np.round(safe["meta"].coef_, 3), "intercept", round(float(safe["meta"].intercept_[0]), 3))
print(confusion_matrix(y, (safe["oof_prob"] >= safe["t"]).astype(int)))


cv meta5 acc 0.85522 t 0.626
meta_coef [[1.234 1.489 0.469 1.17  0.968]] intercept -2.632
[[520  29]
 [100 242]]


In [11]:
submission = pd.DataFrame({"PassengerId": pid, "Survived": (safe["test_prob"] >= safe["t"]).astype(int)})
submission.to_csv(ROOT / "submission.csv", index=False)
print("wrote", ROOT / "submission.csv", safe.get("mode", ""))
submission.head(10)


wrote c:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\Titanic\submission.csv meta5


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0
